In [1]:
import json
import pandas as pd

In [3]:
# get a list of all cluster keywords from exhibition_clusters.json, institution_cluster_summary.json, and artist_cluster_summary.json, and write out a flat list of all possible keywords

with open("../../data/clusters/exhibition_clusters.json", "r") as f:
    exhibition_clusters = json.load(f)

with open("../../data/clusters/institution_cluster_summary.json", "r") as f:
    institution_cluster_summary = json.load(f)

with open("../../data/clusters/artist_cluster_summary.json", "r") as f:
    artist_cluster_summary = json.load(f)

all_keywords = set()
# items to take from the front of each cluster's keyword array (None = take all)
n_exhibition = 1
n_institution = 1
n_artist = 1

def first_n_from(seq, n):
    if n is None:
        return seq
    return seq[:n]

def normalize_keywords_list(klist):
    out = []
    for item in klist:
        if isinstance(item, str):
            out.append(item)
        elif isinstance(item, dict):
            # try common fields for keyword text
            for key in ("word", "label", "keyword", "text", "name"):
                if key in item:
                    out.append(item[key])
                    break
    return out

all_keywords = set()
# exhibitions: labels or keywords
for cluster in exhibition_clusters.get("exhibitions", []):
    labels = cluster.get("labels", []) or cluster.get("keywords", [])
    kws = normalize_keywords_list(labels)
    kws = first_n_from(kws, n_exhibition)
    all_keywords.update(kws)

# institution: current format uses 'numClusters' -> entries with 'groups' lists; also check legacy 'cluster_positions' dicts
for entry in institution_cluster_summary.get("numClusters", []):
    for group in entry.get("groups", []):
        kws = normalize_keywords_list(group.get("keywords", []) or group.get("labels", []))
        kws = first_n_from(kws, n_institution)
        all_keywords.update(kws)
    for cluster_data in entry.get("cluster_positions", {}).values():
        kws = normalize_keywords_list(cluster_data.get("keywords", []))
        kws = first_n_from(kws, n_institution)
        all_keywords.update(kws)

# artist: same pattern as institution
for entry in artist_cluster_summary.get("numClusters", []):
    for group in entry.get("groups", []):
        kws = normalize_keywords_list(group.get("keywords", []) or group.get("labels", []))
        kws = first_n_from(kws, n_artist)
        all_keywords.update(kws)
    for cluster_data in entry.get("cluster_positions", {}).values():
        kws = normalize_keywords_list(cluster_data.get("keywords", []))
        kws = first_n_from(kws, n_artist)
        all_keywords.update(kws)



In [4]:
len(all_keywords)

70

In [7]:
# sort and cast to a list
all_keywords = sorted(list(all_keywords))
keywords_df = pd.DataFrame(all_keywords, columns=["word"])
keywords_df.head()

keywords_df.to_csv("../../data/addl/cluster_keywords.csv", index=False)